<a href="https://colab.research.google.com/github/Patas17/Practica-2-IA/blob/main/simulacion_rutas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Optimización de Rutas de Evacuación ante Inundaciones**

**Asignatura:** Fundamentos de IA

**Equipo:**

*   Cardos Martinez Emilio Raul
*   Maria Jose Bravo Calderon
*   Ceme Carrillo Ximena Monserrat
*   Vargas Rodrigue Roberto Jesus

**Contexto y planeacion**

Tras un desastre natural por inundación, una comunidad requiere evacuar a la población hacia un refugio seguro. En situaciones de emergencia, la ruta geométricamente más corta no siempre resulta ser la más viable: caminos directos pueden atravesar niveles de agua críticos con alto riesgo de atrapamiento o corrientes peligrosas.

El objetivo de este proyecto es implementar, comparar y evaluar algoritmos de búsqueda en grafos para resolver la disyuntiva entre distancia física y seguridad del terreno, demostrando mediante ejecuciones cuantitativas qué estrategia garantiza la evacuación más conveniente.


In [1]:
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ==========================================
# CONSTANTES DE NIVELES DE INUNDACIÓN
# ==========================================
SECO = 0      # Terreno completamente libre y seguro
LEVE = 1      # Inundación baja (transitable con precaución)
MEDIO = 2     # Inundación moderada (alto riesgo)
BLOQUEO = 3   # Inundación severa / Obstáculo (intransitable)

# Diccionario de riesgo base asociado a cada nivel: Riesgo(v)
COSTOS_NIVEL = {
    SECO: 0.0,
    LEVE: 1.0,
    MEDIO: 3.0,
    BLOQUEO: float('inf')
}

print("Imports y constantes cargados correctamente.")

Imports y constantes cargados correctamente.


In [2]:
class TerrenoInundado:
    """
    Representa la cuadrícula de la zona inundada, gestiona las restricciones
    de paso y calcula el costo dinámico C(u, v) = 1.0 + alpha * Riesgo(v).
    """
    def __init__(self, matriz, inicio, refugio, alpha=2.0):
        self.grid = np.array(matriz, dtype=int)
        self.filas, self.columnas = self.grid.shape
        self.inicio = tuple(inicio)
        self.refugio = tuple(refugio)
        self.alpha = float(alpha)

    def en_limites(self, nodo):
        """Verifica si la coordenada (r, c) está dentro de la matriz."""
        r, c = nodo
        return 0 <= r < self.filas and 0 <= c < self.columnas

    def es_transitable(self, nodo):
        """Comprueba que la celda esté dentro de los límites y no sea BLOQUEO (3)."""
        if not self.en_limites(nodo):
            return False
        r, c = nodo
        return self.grid[r, c] != BLOQUEO

    def obtener_riesgo(self, nodo):
        """Retorna el valor Riesgo(v) según el nivel de inundación de la celda."""
        r, c = nodo
        nivel = self.grid[r, c]
        return COSTOS_NIVEL.get(nivel, float('inf'))

    def costo_movimiento(self, u, v):
        """
        Evalúa la fórmula matemática de costo: C(u, v) = 1.0 + alpha * Riesgo(v).
        Si la celda destino v está bloqueada, retorna infinito.
        """
        if not self.es_transitable(v):
            return float('inf')
        return 1.0 + self.alpha * self.obtener_riesgo(v)

    def vecinos(self, nodo):
        """Retorna las celdas adyacentes ortogonales (arriba, abajo, izq, der) que son transitables."""
        r, c = nodo
        direcciones = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        resultado = []
        for dr, dc in direcciones:
            vecino = (r + dr, c + dc)
            if self.es_transitable(vecino):
                resultado.append(vecino)
        return resultado

    def modificar_celda(self, nodo, nuevo_nivel):
        """Permite cambiar el nivel de inundación o bloquear una celda para pruebas."""
        if self.en_limites(nodo):
            self.grid[nodo[0], nodo[1]] = nuevo_nivel

    def clonar(self):
        """Devuelve una copia exacta e independiente del terreno actual."""
        return TerrenoInundado(self.grid.copy(), self.inicio, self.refugio, self.alpha)


def reconstruir_camino(padres, inicio, meta):
    """
    Reconstruye la lista de coordenadas desde el inicio hasta la meta
    usando el diccionario de nodos padres generado por la búsqueda.
    """
    if meta not in padres and meta != inicio:
        return None
    camino = []
    actual = meta
    while actual is not None:
        camino.append(actual)
        actual = padres.get(actual, None)
    camino.reverse()
    return camino if camino and camino[0] == inicio else None


def calcular_metricas_camino(terreno, camino):
    """
    Evalúa una ruta obtenida y calcula:
    - pasos: longitud geométrica del camino.
    - riesgo_acumulado: suma de Riesgo(v) de las celdas pisadas.
    - costo_total: suma de C(u, v) = 1.0 + alpha * Riesgo(v).
    - es_valida: verificación de continuidad y ausencia de bloqueos.
    """
    if not camino or len(camino) == 0:
        return {
            "pasos": 0,
            "riesgo_acumulado": float('inf'),
            "costo_total": float('inf'),
            "es_valida": False
        }

    # Verificar origen, destino y transitabilidad de cada nodo
    if camino[0] != terreno.inicio or camino[-1] != terreno.refugio:
        return {"pasos": 0, "riesgo_acumulado": float('inf'), "costo_total": float('inf'), "es_valida": False}

    pasos = len(camino) - 1
    riesgo_acumulado = 0.0
    costo_total = 0.0

    for i in range(len(camino)):
        nodo_actual = camino[i]
        if not terreno.es_transitable(nodo_actual):
            return {"pasos": 0, "riesgo_acumulado": float('inf'), "costo_total": float('inf'), "es_valida": False}

        # El riesgo acumulado es la suma del riesgo de cada celda visitada
        riesgo_acumulado += terreno.obtener_riesgo(nodo_actual)

        if i > 0:
            nodo_previo = camino[i - 1]
            costo_total += terreno.costo_movimiento(nodo_previo, nodo_actual)

    return {
        "pasos": pasos,
        "riesgo_acumulado": riesgo_acumulado,
        "costo_total": costo_total,
        "es_valida": True
    }

## 2. Estrategias de Búsqueda Implementadas

Para analizar el problema desde distintas perspectivas y distinguir entre encontrar *cualquier* ruta y encontrar una ruta *conveniente*, implementamos tres algoritmos:

1. **BFS (Breadth-First Search / Búsqueda en Anchura):** Explora el terreno nivel por nivel usando una cola FIFO (`deque`). Ignora por completo el nivel de inundación (mientras no sea bloqueo total) y garantiza encontrar el camino con el **menor número de pasos**. Sirve como línea base para demostrar por qué el camino más directo suele ser peligroso.
2. **Dijkstra (Búsqueda de Costo Uniforme):** Utiliza una cola de prioridad (`heapq`) ordenada por el costo acumulado $g(n)$. Evalúa el riesgo de cada celda y garantiza encontrar la ruta más segura y conveniente (menor costo total entre distancia y riesgo).
3. **A* (A-Estrella):** Mejora la eficiencia de Dijkstra sumando al costo acumulado $g(n)$ una heurística admisible $h(n)$ basada en la **Distancia Manhattan** hacia el refugio: $f(n) = g(n) + \vert{}r_1 - r_2\vert{} + \vert{}c_1 - c_2\vert{}$. Encuentra la misma ruta óptima que Dijkstra pero expandiendo menos nodos.

In [3]:
def busqueda_bfs(terreno, inicio, fin):
    """BFS: Encuentra la ruta con menor número de pasos ignorando el nivel de riesgo."""
    cola = deque([inicio])
    came_from = {inicio: None}
    nodos_expandidos = 0

    while cola:
        actual = cola.popleft()
        nodos_expandidos += 1

        if actual == fin:
            break

        for vecino in terreno.obtener_vecinos(*actual):
            if vecino not in came_from:
                came_from[vecino] = actual
                cola.append(vecino)

    camino = reconstruir_camino(came_from, inicio, fin)
    metricas = calcular_metricas_camino(camino, terreno)
    metricas["nodos_expandidos"] = nodos_expandidos
    return camino, metricas

In [4]:
def busqueda_dijkstra(terreno, inicio, fin):
    """Dijkstra: Minimiza el costo acumulado (distancia + riesgo) usando cola de prioridad."""
    pq = [(0.0, inicio)]
    costo_acumulado = {inicio: 0.0}
    came_from = {inicio: None}
    nodos_expandidos = 0

    while pq:
        costo_actual, actual = heapq.heappop(pq)
        nodos_expandidos += 1

        if actual == fin:
            break

        if costo_actual > costo_acumulado[actual]:
            continue

        for vecino in terreno.obtener_vecinos(*actual):
            nuevo_costo = costo_actual + terreno.costo_transicion(*vecino)
            if (
                vecino not in costo_acumulado
                or nuevo_costo < costo_acumulado[vecino]
            ):
                costo_acumulado[vecino] = nuevo_costo
                came_from[vecino] = actual
                heapq.heappush(pq, (nuevo_costo, vecino))

    camino = reconstruir_camino(came_from, inicio, fin)
    metricas = calcular_metricas_camino(camino, terreno)
    metricas["nodos_expandidos"] = nodos_expandidos
    return camino, metricas


def busqueda_astar(terreno, inicio, fin):
    """A*: Usa heurística Manhattan para encontrar la ruta óptima explorando menos celdas."""

    def heuristica(a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    pq = [(heuristica(inicio, fin), 0.0, inicio)]
    g_score = {inicio: 0.0}
    came_from = {inicio: None}
    nodos_expandidos = 0

    while pq:
        f_val, g_val, actual = heapq.heappop(pq)
        nodos_expandidos += 1

        if actual == fin:
            break

        if g_val > g_score[actual]:
            continue

        for vecino in terreno.obtener_vecinos(*actual):
            tentative_g = g_score[actual] + terreno.costo_transicion(*vecino)
            if vecino not in g_score or tentative_g < g_score[vecino]:
                g_score[vecino] = tentative_g
                f_score = tentative_g + heuristica(vecino, fin)
                came_from[vecino] = actual
                heapq.heappush(pq, (f_score, tentative_g, vecino))

    camino = reconstruir_camino(came_from, inicio, fin)
    metricas = calcular_metricas_camino(camino, terreno)
    metricas["nodos_expandidos"] = nodos_expandidos
    return camino, metricas